# 01 — Data Ingestion and Cleaning

## Objective

The objective of this notebook is to ingest, clean, and standardize the raw datasets used in the Hourly Ontario Energy Price (HOEP) forecasting project.

The raw data is obtained from the Independent Electricity System Operator (IESO) and includes three main sources:

- **Hourly Ontario Energy Price (HOEP)**
- **Electricity demand data**
- **Electricity generation by fuel type**

These datasets are provided as yearly CSV files and require several preprocessing steps before they can be used for exploratory analysis and forecasting models.

This notebook prepares the datasets so they can be reliably used in later stages of the project.

---

# Data Processing Workflow

The data ingestion pipeline in this notebook performs the following steps:

### 1. Load raw datasets
The raw HOEP, demand, and generation files are loaded from the `data/raw` directory using **PySpark**.

### 2. Remove metadata rows
Some datasets include metadata lines at the beginning of the file (for example report titles or creation timestamps).  
These rows are skipped during ingestion so that only the actual tabular data is processed.

### 3. Standardize column names
Column names are cleaned and converted to a consistent format (lowercase and underscore naming) to simplify later processing and merging.

### 4. Convert data types
Relevant variables such as:

- hour
- price
- demand
- generation output

are converted to appropriate numeric data types.

### 5. Standardize date formats
The date format differs across datasets and years.  
Dates are normalized to a consistent format so that the datasets can be merged reliably.

### 6. Construct a consistent time index
Electricity market datasets use a reporting structure based on:

- **date**
- **hour (1–24)**

These two variables uniquely identify each hourly observation in the market.

### 7. Validate the datasets
Basic validation checks are performed, including:

- verifying row counts
- checking the range of dates
- verifying that hours range from 1 to 24
- identifying duplicate observations
- detecting missing values

### 8. Align dataset time ranges
The HOEP dataset is available only until **April 30, 2025**.  
To ensure consistency across datasets, the demand and generation datasets are truncated to the same time range.

### 9. Save cleaned datasets
The cleaned datasets are saved in an efficient format for downstream processing and modeling.

---

# Challenge: Daylight Saving Time (DST)

A common challenge in electricity market datasets is handling **Daylight Saving Time (DST)** transitions.

In Ontario, clocks shift twice per year:

- **Spring transition:** clocks move forward by one hour (a 23-hour day)
- **Fall transition:** clocks move backward by one hour (a 25-hour day)

When converting the `date` and `hour` fields into standard timestamps, DST transitions can create:

- duplicated timestamps
- missing timestamps
- misaligned hourly observations

For example, during the spring transition, the hour between **02:00 and 02:59 does not exist**, which may cause two different market hours to map to the same timestamp.

To avoid these issues, this project uses the **market reporting structure (`date`, `hour`) as the primary time index** rather than immediately converting to civil timestamps.  
This ensures that every hourly market observation remains uniquely identified and prevents time alignment errors caused by DST.

---

# Why We Store Data in Parquet Format

After cleaning, the datasets are stored in **Parquet format** instead of CSV.

Parquet is a columnar storage format widely used in big data and distributed computing environments.

It offers several advantages:

### Efficient storage
Parquet compresses data efficiently and reduces disk usage compared to CSV files.

### Faster data processing
Because Parquet is column-oriented, Spark can read only the columns needed for a computation, significantly improving performance.

### Better compatibility with big data tools
Parquet is the standard format used in many data processing frameworks such as:

- Apache Spark
- Hadoop
- cloud data warehouses

### Reliable schema preservation
Unlike CSV files, Parquet preserves data types, which avoids repeated schema inference and ensures consistent downstream processing.

For these reasons, the cleaned datasets are stored as Parquet files in the project pipeline.

---

# Output of This Notebook

At the end of this notebook, the following cleaned datasets are produced:
`data/interim/
hoep_clean.parquet
demand_clean.parquet
generation_clean.parquet`


These datasets will be used in the next stage of the project, where they will be merged into a single modeling dataset and explored through exploratory data analysis (EDA).


# Initial setup

1. Imports

In [609]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
import re
from pathlib import Path
import os

2. Start Spark


In [610]:
spark = (
    SparkSession.builder
    .appName("HOEP_Data_Ingestion")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.5.1


3. Paths

In [611]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Interim data directory: {INTERIM_DIR}")

Project root: /home/milad/projects/HOEP-forcasting-project
Raw data directory: /home/milad/projects/HOEP-forcasting-project/data/raw
Interim data directory: /home/milad/projects/HOEP-forcasting-project/data/interim


3. List files 

In [612]:
for file in sorted(os.listdir(RAW_DATA_DIR)):
    print(file)

PUB_Demand_2023.csv
PUB_Demand_2023.csv:Zone.Identifier
PUB_Demand_2024.csv
PUB_Demand_2024.csv:Zone.Identifier
PUB_Demand_2025.csv
PUB_Demand_2025.csv:Zone.Identifier
PUB_PriceHOEPPredispOR_2023.csv
PUB_PriceHOEPPredispOR_2023_v395.csv:Zone.Identifier
PUB_PriceHOEPPredispOR_2024.csv
PUB_PriceHOEPPredispOR_2024.csv:Zone.Identifier
PUB_PriceHOEPPredispOR_2025 (1).csv:Zone.Identifier
PUB_PriceHOEPPredispOR_2025.csv
generation_2023.csv
generation_2024.csv
generation_2025.csv


# Helper Functions

1. Standardize column names


In [613]:
def standardize_column_names(df):
    clean_cols = []
    for c in df.columns:
        new_c = (
            c.strip()
             .lower()
             .replace(" ", "_")
             .replace("-", "_")
             .replace("/", "_")
        )
        clean_cols.append(new_c)
    return df.toDF(*clean_cols)

2. Read files with metadata (HOEP and market demand files)

In [614]:
def read_file_with_metadata(file_path : Path|str):
    if isinstance(file_path, Path):
        file_path = str(file_path)
    return (
        spark.read
        .option("header", True)
        .option("sep", ",")
        .option("inferSchema", True)
        .option("comment", "\\")
        .csv(str(file_path))
    )

3. Add market key column

In [615]:
def add_market_key(df, date_col="date", hour_col="hour"):
    
    df = df.withColumn(date_col, F.regexp_replace(F.col(date_col), "/", "-"))

    # Detect date format from first non-null row
    sample_row = df.select(date_col).filter(F.col(date_col).isNotNull()).first()
    sample_date = str(sample_row[date_col])

    if re.match(r"^\d{4}-", sample_date):
        date_format = "yyyy-MM-dd"
    else:
        date_format = "dd-MM-yyyy"

    # Parse date safely
    df = df.withColumn("parsed_date", F.to_date(F.col(date_col), date_format))

    # Standardize date string
    df = df.withColumn("date", F.date_format(F.col("parsed_date"), "yyyy-MM-dd"))

    # Ensure hour is integer
    df = df.withColumn(hour_col, F.col(hour_col).cast("int"))

    # Create market key
    df = df.withColumn(
        "market_key",
        F.concat_ws("_H", F.col("date"), F.format_string("%02d", F.col(hour_col)))
    )

    return df.drop("parsed_date", "date_std")

4. Cast numeric columns 


In [616]:
def cast_columns(df, cast_map):
    """
    cast_map example:
    {
        "hour": "int",
        "hoep": "double"
    }
    """
    for col_name, col_type in cast_map.items():
        df = df.withColumn(col_name, F.col(col_name).cast(col_type))
    return df

# HOEP Ingestion


1. HOEP files path

In [617]:
hoep_files = [
    RAW_DATA_DIR / f for f in os.listdir(RAW_DATA_DIR)
    if f.startswith("PUB_PriceHOEP") and f.endswith(".csv")
]


2. Load HOEP files

In [618]:
hoep_dfs = []
for path in hoep_files:
    df = read_file_with_metadata(path)
    df = standardize_column_names(df)
    hoep_dfs.append(df)

print(f"Read {len(hoep_dfs)} HOEP files into DataFrames.")

Read 3 HOEP files into DataFrames.


3. Combine HOEP files

In [619]:
hoep_df = hoep_dfs[0]
for df in hoep_dfs[1:]:
    hoep_df = hoep_df.union(df)

print("HOEP total rows:", hoep_df.count())
hoep_df.show(5)

HOEP total rows: 20424
+----------+----+-----+------------------+------------------+------------------+--------------+------------------+---------+
|      date|hour| hoep|hour_1_predispatch|hour_2_predispatch|hour_3_predispatch|or_10_min_sync|or_10_min_non_sync|or_30_min|
+----------+----+-----+------------------+------------------+------------------+--------------+------------------+---------+
|2023-01-01|   1|14.42|             41.03|              43.0|             39.59|          0.15|              0.15|     0.15|
|2023-01-01|   2|19.21|             47.23|             43.66|              39.6|          0.15|              0.15|     0.15|
|2023-01-01|   3| 14.5|             47.24|              49.0|             39.54|          0.15|              0.15|     0.15|
|2023-01-01|   4|26.26|             44.81|             45.94|             40.27|          0.15|              0.15|     0.15|
|2023-01-01|   5|35.78|              45.0|              45.2|              40.5|          0.15|       

4. Keep required HOEP columns

In [620]:
hoep_df = hoep_df.select(
    "date",
    "hour",
    "hoep",
    "hour_1_predispatch",
    "hour_2_predispatch",
    "hour_3_predispatch"
)

5. Cast HOEP columns and add market key

In [621]:
hoep_df = cast_columns(hoep_df, {
    "hour": "int",
    "hoep": "double",
    "hour_1_predispatch": "double",
    "hour_2_predispatch": "double",
    "hour_3_predispatch": "double"
})

hoep_df = add_market_key(hoep_df)

hoep_df.show(48, truncate=False)

+----------+----+-----+------------------+------------------+------------------+--------------+
|date      |hour|hoep |hour_1_predispatch|hour_2_predispatch|hour_3_predispatch|market_key    |
+----------+----+-----+------------------+------------------+------------------+--------------+
|2023-01-01|1   |14.42|41.03             |43.0              |39.59             |2023-01-01_H01|
|2023-01-01|2   |19.21|47.23             |43.66             |39.6              |2023-01-01_H02|
|2023-01-01|3   |14.5 |47.24             |49.0              |39.54             |2023-01-01_H03|
|2023-01-01|4   |26.26|44.81             |45.94             |40.27             |2023-01-01_H04|
|2023-01-01|5   |35.78|45.0              |45.2              |40.5              |2023-01-01_H05|
|2023-01-01|6   |14.49|49.0              |46.64             |38.75             |2023-01-01_H06|
|2023-01-01|7   |40.2 |39.62             |40.27             |39.62             |2023-01-01_H07|
|2023-01-01|8   |31.48|38.75            

# Demand Ingestion


1. Demand files path

In [622]:
demand_files = [
    RAW_DATA_DIR / f for f in os.listdir(RAW_DATA_DIR)
    if f.startswith("PUB_Demand") and f.endswith(".csv")
]

demand_files

[PosixPath('/home/milad/projects/HOEP-forcasting-project/data/raw/PUB_Demand_2024.csv'),
 PosixPath('/home/milad/projects/HOEP-forcasting-project/data/raw/PUB_Demand_2025.csv'),
 PosixPath('/home/milad/projects/HOEP-forcasting-project/data/raw/PUB_Demand_2023.csv')]

2. Load and Combine Demand files


In [623]:
demand_dfs = []
for path in demand_files:
    df = read_file_with_metadata(path)
    df = standardize_column_names(df)
    demand_dfs.append(df)

print(f"Read {len(demand_dfs)} Demand files into DataFrames.")

demand_df = demand_dfs[0]
for df in demand_dfs[1:]:
    demand_df = demand_df.union(df)

print("Demand total rows:", demand_df.count())
demand_df.show(5)

Read 3 Demand files into DataFrames.
Demand total rows: 26303
+----------+----+-------------+--------------+
|      date|hour|market_demand|ontario_demand|
+----------+----+-------------+--------------+
|2024-01-01|   1|        17091|         14482|
|2024-01-01|   2|        16658|         14180|
|2024-01-01|   3|        16233|         13722|
|2024-01-01|   4|        15909|         13637|
|2024-01-01|   5|        15998|         13697|
+----------+----+-------------+--------------+
only showing top 5 rows



3. Cast demand columns and add market key

In [624]:
demand_df = cast_columns(demand_df, {
    "hour": "int",
    "market_demand": "double",
    "ontario_demand": "double"
})

demand_df= add_market_key(demand_df)


demand_df.show(48, truncate=False)

+----------+----+-------------+--------------+--------------+
|date      |hour|market_demand|ontario_demand|market_key    |
+----------+----+-------------+--------------+--------------+
|2024-01-01|1   |17091.0      |14482.0       |2024-01-01_H01|
|2024-01-01|2   |16658.0      |14180.0       |2024-01-01_H02|
|2024-01-01|3   |16233.0      |13722.0       |2024-01-01_H03|
|2024-01-01|4   |15909.0      |13637.0       |2024-01-01_H04|
|2024-01-01|5   |15998.0      |13697.0       |2024-01-01_H05|
|2024-01-01|6   |16331.0      |13787.0       |2024-01-01_H06|
|2024-01-01|7   |16493.0      |14055.0       |2024-01-01_H07|
|2024-01-01|8   |17486.0      |14481.0       |2024-01-01_H08|
|2024-01-01|9   |17687.0      |14708.0       |2024-01-01_H09|
|2024-01-01|10  |17549.0      |14890.0       |2024-01-01_H10|
|2024-01-01|11  |17884.0      |15240.0       |2024-01-01_H11|
|2024-01-01|12  |18180.0      |15564.0       |2024-01-01_H12|
|2024-01-01|13  |18113.0      |15920.0       |2024-01-01_H13|
|2024-01

4. Drop data after the last day of hoep dataset  

In [625]:
max_hoep_date = hoep_df.select(F.max("date")).collect()[0][0]
print("Last HOEP date:", max_hoep_date)

demand_df = demand_df.filter(
    F.col("date") <= max_hoep_date
)

Last HOEP date: 2025-04-30


# Generation ingestion

1. Generation file paths

In [626]:
generation_files = [
    RAW_DATA_DIR / f for f in os.listdir(RAW_DATA_DIR)
    if f.startswith("generation") and f.endswith(".csv")
]

generation_files

[PosixPath('/home/milad/projects/HOEP-forcasting-project/data/raw/generation_2025.csv'),
 PosixPath('/home/milad/projects/HOEP-forcasting-project/data/raw/generation_2024.csv'),
 PosixPath('/home/milad/projects/HOEP-forcasting-project/data/raw/generation_2023.csv')]

2. Load and Combine Demand files


In [627]:
generation_dfs = []
for path in generation_files:
    df = read_file_with_metadata(path)
    df = standardize_column_names(df)
    generation_dfs.append(df)

print(f"Read {len(generation_dfs)} Demand files into DataFrames.")

generation_df = generation_dfs[0]
for df in generation_dfs[1:]:
    generation_df = generation_df.unionByName(df, allowMissingColumns=True)

print("Demand total rows:", generation_df.count())
generation_df.show(5)

Read 3 Demand files into DataFrames.
Demand total rows: 26304
+----------+----+-------+---+-----+----+-----+-------+-----+------------+
|      date|hour|nuclear|gas|hydro|wind|solar|biofuel|other|total_output|
+----------+----+-------+---+-----+----+-----+-------+-----+------------+
|01/01/2025|   1|  10404|222| 3783|2846|    0|     18|  N/A|       17273|
|01/01/2025|   2|  10402|153| 3872|2849|    0|     19|  N/A|       17295|
|01/01/2025|   3|  10407|153| 3970|2956|    0|     20|  N/A|       17506|
|01/01/2025|   4|  10403|154| 3362|3091|    0|     20|  N/A|       17030|
|01/01/2025|   5|  10400|218| 3672|3096|    0|     19|  N/A|       17405|
+----------+----+-------+---+-----+----+-----+-------+-----+------------+
only showing top 5 rows



3. Cast Generation columns and add market key

In [628]:
generation_df = cast_columns(generation_df, {
    
        "hour": "int",
        "nuclear": "double",
        "gas": "double",
        "hydro": "double",
        "wind": "double",
        "solar": "double",
        "biofuel": "double",
})

generation_df= add_market_key(generation_df)


generation_df.show(48, truncate=False)

+----------+----+-------+------+------+------+-----+-------+-----+------------+--------------+
|date      |hour|nuclear|gas   |hydro |wind  |solar|biofuel|other|total_output|market_key    |
+----------+----+-------+------+------+------+-----+-------+-----+------------+--------------+
|2025-01-01|1   |10404.0|222.0 |3783.0|2846.0|0.0  |18.0   |N/A  |17273       |2025-01-01_H01|
|2025-01-01|2   |10402.0|153.0 |3872.0|2849.0|0.0  |19.0   |N/A  |17295       |2025-01-01_H02|
|2025-01-01|3   |10407.0|153.0 |3970.0|2956.0|0.0  |20.0   |N/A  |17506       |2025-01-01_H03|
|2025-01-01|4   |10403.0|154.0 |3362.0|3091.0|0.0  |20.0   |N/A  |17030       |2025-01-01_H04|
|2025-01-01|5   |10400.0|218.0 |3672.0|3096.0|0.0  |19.0   |N/A  |17405       |2025-01-01_H05|
|2025-01-01|6   |10400.0|394.0 |3315.0|3250.0|0.0  |18.0   |N/A  |17377       |2025-01-01_H06|
|2025-01-01|7   |10400.0|493.0 |3545.0|3166.0|0.0  |21.0   |N/A  |17625       |2025-01-01_H07|
|2025-01-01|8   |10399.0|494.0 |4028.0|3166.0|0.0 

4. Drop exftra rows after the last day of hoep dataset

In [629]:
generation_df = generation_df.filter(
    F.col("date") <= max_hoep_date
)

# Basic Checks

1. check date range

In [630]:
print("HOEP date range")
hoep_df.select(
    F.min("date"),
    F.max("date")
).show()

print("Demand date range")
demand_df.select(
    F.min("date"),
    F.max("date")
).show()

print("Generation date range")
generation_df.select(
    F.min("date"),
    F.max("date")
).show()

HOEP date range
+----------+----------+
| min(date)| max(date)|
+----------+----------+
|2023-01-01|2025-04-30|
+----------+----------+

Demand date range
+----------+----------+
| min(date)| max(date)|
+----------+----------+
|2023-01-01|2025-04-30|
+----------+----------+

Generation date range
+----------+----------+
| min(date)| max(date)|
+----------+----------+
|2023-01-01|2025-04-30|
+----------+----------+



2. Check duplicates

In [631]:
print("HOEP duplicates")
hoep_df.groupBy("date","hour").count().filter(F.col("count") > 1).show()

print("Demand duplicates")
demand_df.groupBy("date","hour").count().filter(F.col("count") > 1).show()

print("Generation duplicates")
generation_df.groupBy("date","hour").count().filter(F.col("count") > 1).show()

HOEP duplicates
+----+----+-----+
|date|hour|count|
+----+----+-----+
+----+----+-----+

Demand duplicates
+----+----+-----+
|date|hour|count|
+----+----+-----+
+----+----+-----+

Generation duplicates
+----+----+-----+
|date|hour|count|
+----+----+-----+
+----+----+-----+



3. Check hour range

In [632]:
print("HOEP hour range")
hoep_df.select(
    F.min("hour"),
    F.max("hour")
).show()

print("Demand hour range")
demand_df.select(
    F.min("hour"),
    F.max("hour")
).show()

print("Generation hour range")
generation_df.select(
    F.min("hour"),
    F.max("hour")
).show()

HOEP hour range
+---------+---------+
|min(hour)|max(hour)|
+---------+---------+
|        1|       24|
+---------+---------+

Demand hour range
+---------+---------+
|min(hour)|max(hour)|
+---------+---------+
|        1|       24|
+---------+---------+

Generation hour range
+---------+---------+
|min(hour)|max(hour)|
+---------+---------+
|        1|       24|
+---------+---------+



4. Check missing value

In [633]:
print("hoep missing values:")
hoep_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in hoep_df.columns
]).show()

print("demand missing values:")
demand_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in demand_df.columns
]).show()

print("generation missing values:")
generation_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in generation_df.columns
]).show()

hoep missing values:
+----+----+----+------------------+------------------+------------------+----------+
|date|hour|hoep|hour_1_predispatch|hour_2_predispatch|hour_3_predispatch|market_key|
+----+----+----+------------------+------------------+------------------+----------+
|   0|   0|   0|                55|                55|                55|         0|
+----+----+----+------------------+------------------+------------------+----------+

demand missing values:
+----+----+-------------+--------------+----------+
|date|hour|market_demand|ontario_demand|market_key|
+----+----+-------------+--------------+----------+
|   0|   0|            0|             0|         0|
+----+----+-------------+--------------+----------+

generation missing values:
+----+----+-------+---+-----+----+-----+-------+-----+------------+----------+
|date|hour|nuclear|gas|hydro|wind|solar|biofuel|other|total_output|market_key|
+----+----+-------+---+-----+----+-----+-------+-----+------------+----------+
|   0

5. Check chronological order

In [634]:
print("hoep chronological order:")
hoep_df.orderBy("date","hour").show(10, truncate=False)
hoep_df  = hoep_df.orderBy("date","hour")

print("demand chronological order:")
demand_df.orderBy("date","hour").show(10, truncate=False)
demand_df = demand_df.orderBy("date","hour")

print("generation chronological order:")
generation_df.orderBy("date","hour").show(10, truncate=False)
generation_df = generation_df.orderBy("date","hour")

hoep chronological order:
+----------+----+-----+------------------+------------------+------------------+--------------+
|date      |hour|hoep |hour_1_predispatch|hour_2_predispatch|hour_3_predispatch|market_key    |
+----------+----+-----+------------------+------------------+------------------+--------------+
|2023-01-01|1   |14.42|41.03             |43.0              |39.59             |2023-01-01_H01|
|2023-01-01|2   |19.21|47.23             |43.66             |39.6              |2023-01-01_H02|
|2023-01-01|3   |14.5 |47.24             |49.0              |39.54             |2023-01-01_H03|
|2023-01-01|4   |26.26|44.81             |45.94             |40.27             |2023-01-01_H04|
|2023-01-01|5   |35.78|45.0              |45.2              |40.5              |2023-01-01_H05|
|2023-01-01|6   |14.49|49.0              |46.64             |38.75             |2023-01-01_H06|
|2023-01-01|7   |40.2 |39.62             |40.27             |39.62             |2023-01-01_H07|
|2023-01-01|8 

6. Check unique time points

In [635]:
print(
    "HOEP unique hours:",
    hoep_df.select("date","hour").distinct().count()
)

print(
    "Demand unique hours:",
    demand_df.select("date","hour").distinct().count()
)

print(
    "Generation unique hours:",
    generation_df.select("date","hour").distinct().count()
)

HOEP unique hours: 20424
Demand unique hours: 20424
Generation unique hours: 20424


# Saved Clean Output ( as parquet)

In [637]:
hoep_df.write.mode("overwrite").parquet(str(INTERIM_DIR / "hoep_clean.parquet"))
demand_df.write.mode("overwrite").parquet(str(INTERIM_DIR / "demand_clean.parquet"))
generation_df.write.mode("overwrite").parquet(str(INTERIM_DIR / "generation_clean.parquet"))

# Export to HDFS

In [ ]:
HDFS_INTERIM = "hdfs://localhost:9000/hoep_project/interim"

hoep_df.write.mode("overwrite").parquet(f"{HDFS_INTERIM}/hoep_clean.parquet")
demand_df.write.mode("overwrite").parquet(f"{HDFS_INTERIM}/demand_clean.parquet")
generation_df.write.mode("overwrite").parquet(f"{HDFS_INTERIM}/generation_clean.parquet")

print("Exported to HDFS:", HDFS_INTERIM)